# VeloceReduction — Reduce one observing night

This notebook is the master workflow for reducing a complete Veloce observing
night.

**Input:** `observations/YYMMDD/`  
**Output:** `reductions/vr_X.Y.Z/YYMMDD/`

Detailed algorithms are implemented in the `velocereduction` Python modules.
This notebook intentionally contains only the main reduction steps.

## 0. Setup

Choose the night and reduction settings.

When run interactively as a notebook, the values below are used directly.
When converted to `reduce_night.py`, the night is supplied on the command line.

In [ ]:
import sys
import argparse

from velocereduction import (
    __version__,
    utils,
    flat,
    tramlines,
    wavelength,
    velocities,
    tellurics,
)


def running_in_notebook():
    """True for an interactive Jupyter notebook, False for reduce_night.py."""
    if 'ipykernel' in sys.modules:
        return True
    else:
        return False


IN_NOTEBOOK = running_in_notebook()

In [ ]:
if IN_NOTEBOOK:

    # -------------------------------------------------------------------------
    # Interactive notebook settings
    # -------------------------------------------------------------------------

    night = '001122'            # Observing night in YYMMDD format,
                                # e.g. 001122, which is the reference night
    # night = '260703'            

    log_level = 'DEBUG'          # DEBUG / INFO / WARNING / ERROR
    diagnostics = 'full'       # none / basic / full

    extraction_mode = 'summed'  # summed / fibre
    overwrite = True

else:

    # -------------------------------------------------------------------------
    # Command-line settings for reduce_night.py
    # -------------------------------------------------------------------------

    parser = argparse.ArgumentParser(
        description='Reduce one complete Veloce observing night.'
    )

    parser.add_argument(
        'night',
        help='Observing night in YYMMDD format, e.g. 001122'
    )

    parser.add_argument(
        '--log-level',
        choices=['DEBUG', 'INFO', 'WARNING', 'ERROR'],
        default='INFO'
    )

    parser.add_argument(
        '--diagnostics',
        choices=['none', 'basic', 'full'],
        default='basic'
    )

    parser.add_argument(
        '--extraction-mode',
        choices=['summed', 'fibre'],
        default='summed'
    )

    parser.add_argument(
        '--overwrite',
        action='store_true'
    )

    args = parser.parse_args()

    night = args.night
    log_level = args.log_level
    diagnostics = args.diagnostics
    extraction_mode = args.extraction_mode
    overwrite = args.overwrite

In [ ]:
config = utils.ReductionConfig(
    night=night,
    log_level=log_level,
    diagnostics=diagnostics,
    extraction_mode=extraction_mode,
    overwrite=overwrite,
)

paths = utils.prepare_reduction(config, version=__version__)
logger = utils.setup_logging(config, paths)

logger.info('Starting VeloceReduction %s for night %s', __version__, night)
logger.info('Diagnostics: %s', diagnostics)
logger.info('Extraction mode: %s', extraction_mode)

print(f'\nNight:             {night}')
print(f'VeloceReduction:   {__version__}')
print(f'Logging:           {log_level}')
print(f'Diagnostics:       {diagnostics}')
print(f'Extraction:        {extraction_mode}')
print(f'Output:            {paths.root}')

## 1. Identify observations

**Input:** raw observations and observing log  
**Output:** night overview and `reduction_input_YYMMDD.txt`

All observations are classified once at the beginning of the reduction.
Later stages use this table rather than repeatedly reading and classifying
raw FITS headers.

In [ ]:
reduction_input = utils.identify_observations(config, paths)

utils.write_reduction_input(reduction_input, config, paths)

if IN_NOTEBOOK:
    display(reduction_input)

## 2. Measure displacement relative to reference night

Measure the displacement of each detector relative to the reference night.

SimTh images are used for all three CCDs, with SimLC providing an additional
registration measurement for CCD2 and CCD3.

In [ ]:
detector_shifts = tramlines.measure_detector_shifts(
    reduction_input,
    config,
    paths,
)

if IN_NOTEBOOK:
    display(detector_shifts)

## 3. Master Flat, nightly tramlines, and Flat response

**Input:** Flat observations + reference tramlines + detector shifts  
**Output:** master Flat, nightly tramline geometry, extracted Flat, response Flat, and blaze function

Create a high-S/N master Flat in detector coordinates and use it to determine
the nightly tramline geometry. The Flat is then extracted along the fitted
tramlines and smoothed along the fibre direction to separate the small-scale
detector response from the smooth illumination and blaze.

In [ ]:
master_flat = flat.create_master_flat(
    reduction_input,
    config,
    paths,
)

In [ ]:
nightly_tramlines = tramlines.fit_nightly_tramlines(
    reduction_input,
    master_flat,
    detector_shifts,
    config,
    paths,
)

if (
    IN_NOTEBOOK
    and config.diagnostics != 'none'
):
    tramlines.show_summary(
        nightly_tramlines
    )

In [ ]:
flat_products = flat.create_flat_products(
    master_flat,
    nightly_tramlines,
    config,
    paths,
)

## 6. Extract wavelength-calibration observations

**Input:** SimLC and FibTh observations  
**Output:** time-stamped extracted calibration spectra

SimLC and FibTh are both used to establish the wavelength solution.

In [ ]:
# simlc_regions = tramlines.find_simlc_regions(
#     processed.simlc,
#     nightly_tramlines,
#     config,
#     paths
# )

# simlc_spectra = extraction.extract_spectra(
#     processed.simlc,
#     nightly_tramlines,
#     config,
#     paths,
#     region='SimLC',
#     product_type='SimLC'
# )

# fibth_spectra = extraction.extract_spectra(
#     processed.fibth,
#     nightly_tramlines,
#     config,
#     paths,
#     region='Science',
#     product_type='FibTh'
# )

## 7. Fit the wavelength solution

**Input:** extracted SimLC + FibTh spectra and their MJD timestamps  
**Output:** wavelength calibration model for the night

In [ ]:
# wavelength_solution = wavelength.fit_wavelength_solution(
#     simlc_spectra,
#     fibth_spectra,
#     config,
#     paths
# )

## 8. Dark Products

In [ ]:
# dark_products = utils.process_darks(
#     processed,
#     config,
#     paths
# ) 

## 9. Extract science spectra

**Input:** processed Science images + tramlines + Flat products + wavelength model  
**Output:** wavelength-calibrated Science spectra

The current default extraction is summed across the Science fibres.
A fibre-resolved extraction can use the same interface in future.

In [ ]:
# science_spectra = extraction.extract_science(
#     processed.science,
#     nightly_tramlines,
#     flat_products,
#     wavelength_solution,
#     config,
#     paths
# )

## 10. Velocities

**Input:** wavelength-calibrated Science spectra  
**Output:** barycentric corrections and first-pass radial velocities

In [ ]:
# science_spectra = velocities.add_barycentric_corrections(
#     science_spectra,
#     config,
#     paths
# )

# science_spectra = velocities.measure_initial_rvs(
#     science_spectra,
#     template='solar',
#     config=config,
#     paths=paths
# )

## 11. B-star and telluric products

**Input:** B-star observations  
**Output:** extracted B-star spectra and telluric measurements

In [ ]:
# bstar_spectra = extraction.extract_science(
#     processed.bstars,
#     nightly_tramlines,
#     flat_products,
#     wavelength_solution,
#     config,
#     paths,
#     product_type='Bstar'
# )

# telluric_products = tellurics.measure_tellurics(
#     bstar_spectra,
#     config,
#     paths
# )

## 12. Final products and reduction summary

Write the final Science FITS products, retained diagnostic figures, and
human-readable summary of the night.

In [ ]:
# utils.write_science_products(
#     science_spectra,
#     config,
#     paths
# )

# summary = utils.create_reduction_summary(
#     reduction_input=reduction_input,
#     detector_shifts=detector_shifts,
#     tramlines=nightly_tramlines,
#     wavelength_solution=wavelength_solution,
#     science_spectra=science_spectra,
#     config=config,
#     paths=paths
# )

# utils.write_reduction_summary(summary, config, paths)

In [ ]:
logger.info('Reduction completed successfully.')

print()
print('Reduction complete')
print(f'Night:    {night}')
print(f'Version:  {__version__}')
print(f'Products: {paths.root}')
print(f'Summary:  {paths.reduction_summary}')